# 🧠 Exhibition Connector — RAG Base Model Colab

این نوت‌بوک یک **مدل پایه RAG** برای پروژه نمایشگاه است.

ویژگی‌ها:
- بدون GPU
- بدون HF token
- بدون LangChain سنگین
- دانلود داده شرکت‌ها از API/JSON
- ساخت Retriever محلی با TF‑IDF
- تست پرسش‌ها
- مقایسه با Vercel API

این نسخه برای آموزش، دیباگ و تست پایه مناسب است.


In [ ]:
# نصب سبک
!pip install -q requests pandas scikit-learn


In [ ]:
import re, json, requests, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

VERCEL_API_BASE = "https://vercel-app-amber-five.vercel.app"
COMPANIES_JSON_URL = "https://sosa123456-exhibition-connector-rag2-static.static.hf.space/data/companies.json"

print("Vercel API:", VERCEL_API_BASE)
print("Companies JSON:", COMPANIES_JSON_URL)


## 1) بارگذاری داده شرکت‌ها

In [ ]:
resp = requests.get(COMPANIES_JSON_URL, timeout=60)
resp.raise_for_status()
companies = resp.json()
print("تعداد شرکت‌ها:", len(companies))
companies[0]


## 2) نرمال‌سازی فارسی و ساخت متن RAG

In [ ]:
def normalize_fa(text):
    text = str(text or "")
    repl = {"ي":"ی", "ك":"ک", "ۀ":"ه", "ة":"ه", "أ":"ا", "إ":"ا", "ؤ":"و", "\u200c":" ", "\u200f":" ", "\ufeff":" "}
    for a,b in repl.items():
        text = text.replace(a,b)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def record_text(r):
    fields = r.get("fields") or {}
    parts = [
        r.get("company", ""), r.get("website", ""), r.get("activity", ""),
        r.get("category", ""), r.get("hall", ""), r.get("booth", ""), r.get("text", ""),
        " ".join(str(v) for v in fields.values())
    ]
    return normalize_fa("\n".join(parts))

texts = [record_text(r) for r in companies]
print(texts[0][:500])


## 3) ساخت Retriever پایه با TF‑IDF

In [ ]:
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), max_features=90000, sublinear_tf=True)
X = vectorizer.fit_transform(texts)
print("matrix:", X.shape)


## 4) تابع جستجوی پایه RAG

In [ ]:
def search_local(question, k=5):
    q = normalize_fa(question)
    qv = vectorizer.transform([q])
    scores = cosine_similarity(qv, X).ravel()
    idxs = scores.argsort()[::-1][:k]
    rows = []
    for rank, idx in enumerate(idxs, start=1):
        r = companies[int(idx)]
        rows.append({
            "rank": rank,
            "score": float(scores[idx]),
            "company": r.get("company"),
            "activity": r.get("activity"),
            "category": r.get("category"),
            "hall": r.get("hall") or "—",
            "booth": r.get("booth") or "—",
            "website": r.get("website") or "—",
        })
    return pd.DataFrame(rows)

def answer_local(question, k=5):
    df = search_local(question, k)
    lines = [f"پاسخ پایه برای: {question}"]
    for _, row in df.iterrows():
        lines.append(f"{row['rank']}. {row['company']} | سالن/غرفه: {row['hall']}/{row['booth']} | سایت: {row['website']}")
    return "\n".join(lines), df


## 5) تست چند پرسش

In [ ]:
queries = [
    "وب‌سایت شرکت پریسماتک چیست؟",
    "مواد شیمیایی تصفیه آب",
    "شرکت‌های مرتبط با ابزار دقیق کدامند؟",
    "سالن 31B",
    "چطور با مترو به نمایشگاه بین‌المللی تهران برویم؟",
]

for q in queries:
    print("="*80)
    ans, df = answer_local(q, 5)
    print(ans)
    display(df)


## 6) مقایسه با Vercel API

In [ ]:
def search_vercel(question, k=5):
    r = requests.get(f"{VERCEL_API_BASE}/api/search", params={"q": question, "limit": k}, timeout=45)
    r.raise_for_status()
    data = r.json()
    return data

q = "وب‌سایت شرکت پریسماتک چیست؟"
remote = search_vercel(q, 5)
print(remote.get("answer"))
pd.DataFrame(remote.get("results", []))[['company','hall','booth','website','score']].head()


## 7) تست Scraper API

In [ ]:
scrape = requests.get(f"{VERCEL_API_BASE}/api/scrape", params={"url": "https://prismatech.ir/"}, timeout=60)
scrape.raise_for_status()
info = scrape.json()["scraped"]
print("title:", info.get("title"))
print("emails:", info.get("emails"))
print("phones:", info.get("phones")[:3])
print("snippet:", info.get("textSnippet", "")[:300])


## 8) معیارهای پایه برای توسعه بعدی

اگر بخواهید مدل بهتر شود:
- reranker اضافه کنید
- stopword فارسی دقیق‌تر بسازید
- embedding sentence-transformers اضافه کنید
- دیتابیس مرکزی برای scrape updates اضافه کنید
- W&B را از سمت Vercel backend لاگ کنید
